# Compare diarization models on full test.json

Runs inference + DER/JER metrics for:

- Fine-tuned MoWiSS model (local `.nemo`/`.ckpt`)
- NVIDIA base pretrained model (`nvidia/diar_streaming_sortformer_4spk-v2`)

Outputs are saved under `inference_compare_fulltest/`.


In [1]:
from __future__ import annotations

import json
import os
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import importlib
if not hasattr(torch, "version"):
    torch.version = importlib.import_module("torch.version")
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate, JaccardErrorRate, DiarizationPurity, DiarizationCoverage

from nemo.collections.asr.models import SortformerEncLabelModel

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    tqdm = None

PROJECT_ROOT = Path.cwd()

# Manifest to evaluate (full)
TEST_MANIFEST = PROJECT_ROOT / "dataset/manifests/test.json"

# Models
# If you want to test a different fine-tuned checkpoint, change this path.
FINETUNED_MODEL_REF = r"experiments/sortformer/sortformer_streaming_4spk_v2/version_0/checkpoints/sortformer_streaming_4spk_v2.nemo"
BASE_MODEL_REF = "nvidia/diar_streaming_sortformer_4spk-v2"

# Output root
OUT_ROOT = PROJECT_ROOT / "inference_compare_fulltest"

COLLAR = 0.25
SKIP_OVERLAP = True

# Inference config
BATCH_SIZE = 1
NUM_WORKERS = 0  # keep 0 on Windows to avoid multiprocessing pickling errors
VERBOSE = False

# Speed/resume knobs
LIMIT_FILES: Optional[int] = None  # set e.g. 200 for a quick smoke test
SHUFFLE_BEFORE_LIMIT = False
RERUN_IF_RTTM_EXISTS = False
SAVE_EVERY_N = 25  # write partial metrics every N files

print("Project root:", PROJECT_ROOT)
print("Test manifest:", TEST_MANIFEST)
print("Out root:", OUT_ROOT)
print("CUDA available:", torch.cuda.is_available())


c:\Users\hieuh\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


Project root: d:\FPT\DSP\code\project
Test manifest: d:\FPT\DSP\code\project\dataset\manifests\test.json
Out root: d:\FPT\DSP\code\project\inference_compare_fulltest
CUDA available: True


In [2]:
def load_manifest(manifest_path: Path) -> List[Dict[str, Any]]:
    entries: List[Dict[str, Any]] = []
    with manifest_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            entries.append(json.loads(line))
    return entries


def diar_to_rttm_lines(recording_id: str, diar_lines: List[str]) -> List[str]:
    out: List[str] = []
    for line in diar_lines:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        start = float(parts[0])
        end = float(parts[1])
        speaker = parts[2]
        duration = max(0.0, end - start)
        if duration <= 0:
            continue
        out.append(
            f"SPEAKER {recording_id} 1 {start:.3f} {duration:.3f} <NA> <NA> {speaker} <NA> <NA>"
        )
    return out


def load_rttm_as_annotation(rttm_path: Path, fallback_uri: str) -> Annotation:
    annotation = Annotation(uri=fallback_uri)
    if not rttm_path.exists():
        return annotation
    with rttm_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or not line.startswith("SPEAKER"):
                continue
            parts = line.split()
            if len(parts) < 8:
                continue
            start = float(parts[3])
            duration = float(parts[4])
            speaker = parts[7]
            if duration <= 0:
                continue
            annotation[Segment(start, start + duration)] = speaker
    return annotation


def safe_metric(metric_obj, ref: Annotation, hyp: Annotation) -> Optional[float]:
    try:
        return float(metric_obj(ref, hyp) * 100.0)
    except ZeroDivisionError:
        return None


def mean_or_none(values: List[Optional[float]]) -> Optional[float]:
    valid = [x for x in values if x is not None]
    if not valid:
        return None
    return float(sum(valid) / len(valid))


def load_model(model_ref: str, device: str) -> SortformerEncLabelModel:
    model_path = Path(model_ref)
    if model_path.exists():
        model = SortformerEncLabelModel.restore_from(str(model_path))
    else:
        model = SortformerEncLabelModel.from_pretrained(model_ref)
    model.eval()
    if device == "cuda":
        model = model.cuda()
    return model


def _iter_entries(entries: List[Dict[str, Any]]):
    if tqdm is None:
        return entries
    return tqdm(entries)


In [3]:
all_entries = load_manifest(TEST_MANIFEST)

validated: List[Dict[str, Any]] = []
for e in all_entries:
    nspk = int(e.get("num_speakers", 0))
    if nspk not in (1, 2, 3, 4):
        continue
    audio_fp = Path(str(e.get("audio_filepath", "")))
    rttm_fp = Path(str(e.get("rttm_filepath", "")))
    if not audio_fp.exists() or not rttm_fp.exists():
        continue
    validated.append(e)

if SHUFFLE_BEFORE_LIMIT and LIMIT_FILES is not None:
    import random

    rng = random.Random(42)
    rng.shuffle(validated)

entries = validated[:LIMIT_FILES] if LIMIT_FILES is not None else validated

print("Total entries in test.json:", len(all_entries))
print("Validated entries (exists on disk):", len(validated))
print("Entries to evaluate:", len(entries))
print("Speaker-count distribution:")
dist = defaultdict(int)
for e in entries:
    dist[int(e["num_speakers"])] += 1
for k in (1, 2, 3, 4):
    print(f"  {k}: {dist[k]}")


Total entries in test.json: 5464
Validated entries (exists on disk): 5464
Entries to evaluate: 5464
Speaker-count distribution:
  1: 1954
  2: 1386
  3: 1138
  4: 986


In [4]:
@dataclass
class EvalOutputs:
    metrics: Dict[str, Any]
    summary: List[Dict[str, Any]]


def evaluate_model(
    *,
    model_ref: str,
    model_name: str,
    entries: List[Dict[str, Any]],
    out_dir: Path,
    collar: float,
    skip_overlap: bool,
    batch_size: int,
    num_workers: int,
    rerun_if_rttm_exists: bool,
    save_every_n: int,
) -> EvalOutputs:
    out_dir.mkdir(parents=True, exist_ok=True)
    out_rttm_dir = out_dir / "rttm"
    out_rttm_dir.mkdir(parents=True, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n[{model_name}] device={device}")
    print(f"[{model_name}] loading model: {model_ref}")
    model = load_model(model_ref, device)

    global_der = DiarizationErrorRate(collar=collar, skip_overlap=skip_overlap)
    global_jer = JaccardErrorRate(collar=collar, skip_overlap=skip_overlap)
    global_purity = DiarizationPurity(collar=collar, skip_overlap=skip_overlap)
    global_coverage = DiarizationCoverage(collar=collar, skip_overlap=skip_overlap)

    by_spk_der: Dict[int, List[Optional[float]]] = defaultdict(list)
    by_spk_jer: Dict[int, List[Optional[float]]] = defaultdict(list)
    by_spk_purity: Dict[int, List[Optional[float]]] = defaultdict(list)
    by_spk_coverage: Dict[int, List[Optional[float]]] = defaultdict(list)
    by_spk_count_acc: Dict[int, List[int]] = defaultdict(list)
    by_spk_count_abs_err: Dict[int, List[int]] = defaultdict(list)

    summary: List[Dict[str, Any]] = []
    per_file_metrics: List[Dict[str, Any]] = []

    partial_metrics_path = out_dir / "metrics_der_jer.partial.json"
    partial_summary_path = out_dir / "summary.partial.json"

    start_t = time.time()
    iterable = _iter_entries(entries)

    for idx, entry in enumerate(iterable, start=1):
        audio_fp = Path(str(entry["audio_filepath"]))
        gt_rttm = Path(str(entry["rttm_filepath"]))
        rec_id = audio_fp.stem
        gt_nspk = int(entry["num_speakers"])
        out_rttm = out_rttm_dir / f"{rec_id}.rttm"

        infer_error = None
        rttm_lines: List[str] = []
        ran_infer = False

        if out_rttm.exists() and not rerun_if_rttm_exists:
            # Reuse previous hypothesis (resume friendly)
            try:
                rttm_lines = [ln for ln in out_rttm.read_text(encoding="utf-8").splitlines() if ln.strip()]
            except Exception as e:
                infer_error = f"read_rttm_error: {e}"
        else:
            try:
                diar_outputs = model.diarize(
                    audio=[str(audio_fp)],
                    batch_size=batch_size,
                    num_workers=num_workers,
                    verbose=VERBOSE,
                )
                diar_lines = diar_outputs[0] if diar_outputs else []
                rttm_lines = diar_to_rttm_lines(rec_id, diar_lines)
                out_rttm.write_text(
                    "\n".join(rttm_lines) + ("\n" if rttm_lines else ""),
                    encoding="utf-8",
                )
                ran_infer = True
            except Exception as e:
                infer_error = str(e)
                out_rttm.write_text("", encoding="utf-8")

        pred_labels = sorted({ln.split()[7] for ln in rttm_lines if ln.strip() and ln.startswith("SPEAKER") and len(ln.split()) >= 8})
        pred_nspk = len(pred_labels)

        summary_item: Dict[str, Any] = {
            "audio_filepath": str(audio_fp),
            "num_speakers_gt": gt_nspk,
            "num_speakers_pred": pred_nspk,
            "pred_speakers": pred_labels,
            "output_rttm": str(out_rttm),
            "ran_inference": bool(ran_infer),
        }
        if infer_error:
            summary_item["error"] = infer_error
        summary.append(summary_item)

        ref = load_rttm_as_annotation(gt_rttm, fallback_uri=rec_id)
        hyp = load_rttm_as_annotation(out_rttm, fallback_uri=rec_id)

        file_der_metric = DiarizationErrorRate(collar=collar, skip_overlap=skip_overlap)
        file_jer_metric = JaccardErrorRate(collar=collar, skip_overlap=skip_overlap)
        file_purity_metric = DiarizationPurity(collar=collar, skip_overlap=skip_overlap)
        file_coverage_metric = DiarizationCoverage(collar=collar, skip_overlap=skip_overlap)
        file_der = safe_metric(file_der_metric, ref, hyp)
        file_jer = safe_metric(file_jer_metric, ref, hyp)
        file_purity = safe_metric(file_purity_metric, ref, hyp)
        file_coverage = safe_metric(file_coverage_metric, ref, hyp)
        safe_metric(global_der, ref, hyp)
        safe_metric(global_jer, ref, hyp)
        safe_metric(global_purity, ref, hyp)
        safe_metric(global_coverage, ref, hyp)

        abs_err = abs(pred_nspk - gt_nspk)
        exact_match = int(pred_nspk == gt_nspk)

        by_spk_der[gt_nspk].append(file_der)
        by_spk_jer[gt_nspk].append(file_jer)
        by_spk_purity[gt_nspk].append(file_purity)
        by_spk_coverage[gt_nspk].append(file_coverage)
        by_spk_count_acc[gt_nspk].append(exact_match)
        by_spk_count_abs_err[gt_nspk].append(abs_err)

        per_file_metrics.append(
            {
                "audio_filepath": str(audio_fp),
                "recording_id": rec_id,
                "num_speakers": gt_nspk,
                "pred_num_speakers": pred_nspk,
                "speaker_count_abs_error": abs_err,
                "speaker_count_exact_match": bool(exact_match),
                "DER": file_der,
                "JER": file_jer,
                "purity": file_purity,
                "coverage": file_coverage,
                "error": infer_error,
            }
        )

        if save_every_n > 0 and (idx % save_every_n == 0):
            partial = {
                "overall": {
                    "DER": None,
                    "JER": None,
                    "speaker_count_accuracy": None,
                    "speaker_count_mae": None,
                    "num_files": len(per_file_metrics),
                },
                "per_file": per_file_metrics,
            }
            partial_metrics_path.write_text(json.dumps(partial, ensure_ascii=False, indent=2), encoding="utf-8")
            partial_summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    try:
        overall_der = float(abs(global_der) * 100.0)
    except ZeroDivisionError:
        overall_der = None
    try:
        overall_jer = float(abs(global_jer) * 100.0)
    except ZeroDivisionError:
        overall_jer = None
    try:
        overall_purity = float(abs(global_purity) * 100.0)
    except ZeroDivisionError:
        overall_purity = None
    try:
        overall_coverage = float(abs(global_coverage) * 100.0)
    except ZeroDivisionError:
        overall_coverage = None

    overall_count_acc = sum(int(x["speaker_count_exact_match"]) for x in per_file_metrics) / len(per_file_metrics)
    overall_count_mae = sum(int(x["speaker_count_abs_error"]) for x in per_file_metrics) / len(per_file_metrics)

    by_num_speakers: Dict[str, Dict[str, Any]] = {}
    for nspk in (1, 2, 3, 4):
        count_acc_list = by_spk_count_acc.get(nspk, [])
        mae_list = by_spk_count_abs_err.get(nspk, [])
        by_num_speakers[str(nspk)] = {
            "DER": mean_or_none(by_spk_der.get(nspk, [])),
            "JER": mean_or_none(by_spk_jer.get(nspk, [])),
            "purity": mean_or_none(by_spk_purity.get(nspk, [])),
            "coverage": mean_or_none(by_spk_coverage.get(nspk, [])),
            "speaker_count_accuracy": (sum(count_acc_list) / len(count_acc_list)) if count_acc_list else None,
            "speaker_count_mae": (sum(mae_list) / len(mae_list)) if mae_list else None,
            "num_files": len(count_acc_list),
        }

    metrics_out: Dict[str, Any] = {
        "overall": {
            "DER": overall_der,
            "JER": overall_jer,
            "purity": overall_purity,
            "coverage": overall_coverage,
            "speaker_count_accuracy": overall_count_acc,
            "speaker_count_mae": overall_count_mae,
            "num_files": len(per_file_metrics),
        },
        "by_num_speakers": by_num_speakers,
        "per_file": per_file_metrics,
    }

    summary_path = out_dir / "summary.json"
    metrics_path = out_dir / "metrics_der_jer.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    metrics_path.write_text(json.dumps(metrics_out, ensure_ascii=False, indent=2), encoding="utf-8")

    elapsed = time.time() - start_t
    print(f"[{model_name}] done. files={len(per_file_metrics)} elapsed_min={elapsed/60.0:.1f}")
    print(f"[{model_name}] overall DER={overall_der} JER={overall_jer} purity={overall_purity} coverage={overall_coverage} count_acc={overall_count_acc:.4f} count_mae={overall_count_mae:.4f}")

    return EvalOutputs(metrics=metrics_out, summary=summary)


In [5]:
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# If the fine-tuned model path does not exist, list candidates.
ft_path = Path(FINETUNED_MODEL_REF)
if not ft_path.exists():
    print("WARNING: FINETUNED_MODEL_REF not found:", FINETUNED_MODEL_REF)
    candidates = sorted(PROJECT_ROOT.glob("experiments/**/*.nemo"))
    print("Found .nemo candidates:")
    for p in candidates:
        print(" -", p)
else:
    print("Fine-tuned model:", ft_path)

print("Base model:", BASE_MODEL_REF)


Fine-tuned model: experiments\sortformer\sortformer_streaming_4spk_v2\version_0\checkpoints\sortformer_streaming_4spk_v2.nemo
Base model: nvidia/diar_streaming_sortformer_4spk-v2


In [6]:
finetuned_out = evaluate_model(
    model_ref=FINETUNED_MODEL_REF,
    model_name="mowiss_finetuned",
    entries=entries,
    out_dir=OUT_ROOT / "mowiss_finetuned",
    collar=COLLAR,
    skip_overlap=SKIP_OVERLAP,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    rerun_if_rttm_exists=RERUN_IF_RTTM_EXISTS,
    save_every_n=SAVE_EVERY_N,
)



[mowiss_finetuned] device=cuda
[mowiss_finetuned] loading model: experiments/sortformer/sortformer_streaming_4spk_v2/version_0/checkpoints/sortformer_streaming_4spk_v2.nemo


[NeMo W 2026-03-13 20:58:41 sortformer_diar_models:94] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: D:\FPT\DSP\code\project\dataset\manifests\train.json
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 30
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 1
    shuffle: true
    num_workers: 0
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 10
    max_duration: 30
    batch_duration: 80
    quadratic_duration: 200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-03-13 20:58:41 sortformer_diar_models:94] If you in

[NeMo I 2026-03-13 20:58:48 modelPT:490] Model SortformerEncLabelModel was successfully restored from d:\FPT\DSP\code\project\experiments\sortformer\sortformer_streaming_4spk_v2\version_0\checkpoints\sortformer_streaming_4spk_v2.nemo.


100%|██████████| 5464/5464 [04:50<00:00, 18.84it/s]


[mowiss_finetuned] done. files=5464 elapsed_min=4.8
[mowiss_finetuned] overall DER=4.700895222794572 JER=4.9830663187267135 purity=98.43262106719813 coverage=97.00214366554955 count_acc=0.7906 count_mae=0.2193


In [7]:
base_out = evaluate_model(
    model_ref=BASE_MODEL_REF,
    model_name="nvidia_base",
    entries=entries,
    out_dir=OUT_ROOT / "nvidia_base",
    collar=COLLAR,
    skip_overlap=SKIP_OVERLAP,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    rerun_if_rttm_exists=RERUN_IF_RTTM_EXISTS,
    save_every_n=SAVE_EVERY_N,
)



[nvidia_base] device=cuda
[nvidia_base] loading model: nvidia/diar_streaming_sortformer_4spk-v2


[NeMo W 2026-03-13 21:03:44 sortformer_diar_models:94] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    pin_memory: true
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-03-13 21:03:44 sortformer_diar_models:94] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: null
    is_tarred: false
    tarred_audio_filepaths: null
    sample_rate: 16

[NeMo I 2026-03-13 21:03:50 modelPT:490] Model SortformerEncLabelModel was successfully restored from C:\Users\hieuh\.cache\huggingface\hub\models--nvidia--diar_streaming_sortformer_4spk-v2\snapshots\6dbf0d69730bfee097056692b86525a0a23b32f9\diar_streaming_sortformer_4spk-v2.nemo.


100%|██████████| 5464/5464 [03:27<00:00, 26.39it/s]


[nvidia_base] done. files=5464 elapsed_min=3.5
[nvidia_base] overall DER=11.218544087990638 JER=18.600641071576966 purity=99.01997330523928 coverage=90.947230228488 count_acc=0.7376 count_mae=0.2886


In [8]:
def summarize(metrics: Dict[str, Any], name: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    o = metrics.get("overall", {})
    rows.append(
        {
            "model": name,
            "group": "overall",
            "DER": o.get("DER"),
            "JER": o.get("JER"),
            "cpWER": o.get("cpWER"),
            "purity": o.get("purity"),
            "coverage": o.get("coverage"),
            "num_files": o.get("num_files"),
        }
    )
    by = metrics.get("by_num_speakers", {})
    for k in ("1", "2", "3", "4"):
        m = by.get(k, {})
        rows.append(
            {
                "model": name,
                "group": f"{k}_spk",
                "DER": m.get("DER"),
                "JER": m.get("JER"),
                "cpWER": m.get("cpWER"),
                "purity": m.get("purity"),
                "coverage": m.get("coverage"),
                "num_files": m.get("num_files"),
            }
        )
    return rows


rows = []
rows += summarize(finetuned_out.metrics, "model_finetuned")
rows += summarize(base_out.metrics, "nvidia_base")

try:
    import pandas as pd

    df = pd.DataFrame(rows)
    df = df[["model", "group", "DER", "JER", "cpWER", "purity", "coverage", "num_files"]]
    display(df)
except Exception:
    print(json.dumps(rows, indent=2, ensure_ascii=False))


def delta(a: Optional[float], b: Optional[float]) -> Optional[float]:
    if a is None or b is None:
        return None
    return float(a - b)


ft_overall = finetuned_out.metrics["overall"]
base_overall = base_out.metrics["overall"]

print("Delta (finetuned - base), overall:")
print("  DER:", delta(ft_overall.get("DER"), base_overall.get("DER")))
print("  JER:", delta(ft_overall.get("JER"), base_overall.get("JER")))
print("  cpWER:", delta(ft_overall.get("cpWER"), base_overall.get("cpWER")))
print("  purity:", delta(ft_overall.get("purity"), base_overall.get("purity")))
print("  coverage:", delta(ft_overall.get("coverage"), base_overall.get("coverage")))


,model,group,DER,JER,cpWER,purity,coverage,num_files
0,model_finetuned,overall,4.700895,4.983066,None,98.432621,97.002144,5464
1,model_finetuned,1_spk,1.561576,0.219071,None,99.386432,99.728862,1954
2,model_finetuned,2_spk,9.212755,5.592657,None,98.098364,97.373079,1386
3,model_finetuned,3_spk,8.067778,5.512192,None,98.030699,95.909545,1138
4,model_finetuned,4_spk,7.980898,5.271483,None,97.349297,95.642616,986
5,nvidia_base,overall,11.218544,18.600641,None,99.019973,90.947230,5464
6,nvidia_base,1_spk,12.398384,11.141476,None,99.554222,86.913608,1954
7,nvidia_base,2_spk,16.397033,18.299880,None,99.302907,89.516676,1386
8,nvidia_base,3_spk,16.838709,19.910244,None,98.717213,90.167072,1138
9,nvidia_base,4_spk,17.288157,21.746479,None,98.037354,90.899744,986


Delta (finetuned - base), overall:
  DER: -6.517648865196066
  JER: -13.617574752850253
  cpWER: None
  purity: -0.5873522380411487
  coverage: 6.054913437061558
